# Solar filament segmentation: end-to-end demo

This notebook reproduces our submission for the Solar Filament Segmentation Challenge 2026 from the
stored weights (no retraining needed), and optionally retrains every model from scratch.

Pipeline (details in `report/report.pdf`):
1. **Pre-processing**: 2048x2048 grayscale H-alpha JPEG, either area-downsampled to 1024x1024 or kept at
   native resolution (depending on the ensemble member); normalisation `(x/255 - 0.45) / 0.25`. Training
   target = *soft consensus mask*: per pixel, the fraction of annotators whose polygons cover it.
2. **Segmentation**: ensemble of U-Nets (5 fold models per member), 4-flip test-time augmentation,
   probability maps upsampled to 2048x2048 and averaged.
3. **Instances**: two YOLO-seg detectors (YOLO11s and YOLO11m, trained on every annotator's polygons) propose
   filament instances; their predictions are merged by mask IoU with averaged confidence (`src/yolo_ens.py`).
   Each merged instance is intersected with the U-Net foreground, painted in confidence order (disjoint
   masks) and dropped when small or weakly supported by the U-Net (`src/fuse.py`). A U-Net-only fallback (thresholded
   probability map, fragments within a small gap grouped) is in `src/postproc.py`.
4. **Output**: disjoint binary masks encoded as COCO RLE (`submission.csv`).

All code lives in `src/` as small modules; this notebook only orchestrates them.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO / "src"))
# Competition data: put (or symlink) the extracted Kaggle data at data/MAGFiLO_1.0_Kaggle_2026
DATA = REPO / "data" / "MAGFiLO_1.0_Kaggle_2026"
assert DATA.exists(), f"expected competition data at {DATA}"
CFG = json.loads((REPO / "configs" / "final.json").read_text())
SMOKE = int(os.environ.get("DEMO_SMOKE", "0"))  # >0: only that many images per step (quick check of the pipeline)
TRAIN_FROM_SCRATCH = False   # True: retrain every model (about 11 GPU-hours on one consumer GPU)
print(json.dumps(CFG, indent=1))

## 1. Pre-processing
Creates `data/cache1024/{train,test}/<image>.npz` (1024px image and soft consensus mask).

In [ ]:
import prep
if not (REPO / "data" / "cache1024" / "test").exists() or len(list((REPO / "data" / "cache1024" / "test").glob("*.npz"))) < 180:
    prep.main()
print(len(list((REPO / "data" / "cache1024" / "train").glob("*.npz"))), "train /",
      len(list((REPO / "data" / "cache1024" / "test").glob("*.npz"))), "test images cached")

## 2. Training (optional)
Folds are split by physical observation (all annotator copies of an image stay in the same fold).
Each U-Net: 30 epochs, 512px random crops (60% centred on filament pixels), flips/rot90/gamma/blur
augmentation, BCE + soft-Dice loss on the soft consensus target, AdamW (lr 5e-4, one-cycle), AMP.
Set `TRAIN_FROM_SCRATCH = True` above to run it; otherwise the stored weights in `weights/` are used.

In [ ]:
if TRAIN_FROM_SCRATCH:
    for m in CFG["unet_members"]:
        env = dict(os.environ, RES=str(m.get("res", 1024)))  # 2048 = train on native-resolution images
        if m.get("res", 1024) != 1024:
            subprocess.run([sys.executable, "src/prep.py"], check=True, cwd=REPO, env=env)
        for f in range(5):
            subprocess.run([sys.executable, "src/train.py", "--fold", str(f), "--enc", m["enc"], "--epochs", "30",
                            "--out", f"runs/{m['name']}_f{f}"], check=True, cwd=REPO, env=env)
            # export to the weights/ layout used below (fp16 state dict)
            import torch
            sd = torch.load(REPO / f"runs/{m['name']}_f{f}/last.pt", map_location="cpu")
            torch.save({k: v.half() for k, v in sd.items()}, REPO / "weights" / f"{m['name']}_f{f}.pt")
    if CFG.get("yolo"):
        import shutil
        # full-data detectors (used for the test set) and fold detectors (used only for the out-of-fold evaluation)
        for f in [-1] + CFG["yolo"].get("fold_models", []):
            split = "all" if f < 0 else f"f{f}"
            subprocess.run([sys.executable, "src/yolo_data.py", "--fold", str(f), "--out", f"yolo_ds{split}"], check=True, cwd=REPO)
            for ym in CFG["yolo"]["models"]:
                subprocess.run([sys.executable, "src/yolo_run.py", "--fold", str(f), "--model", ym["base"], "--epochs", "60",
                                "--batch", str(ym.get("batch", 8)), "--tag", f"{ym['name']}_{split}", "--ds", f"yolo_ds{split}"],
                               check=True, cwd=REPO)
                best = REPO / "runs" / "segment" / "yolo_runs" / f"{ym['name']}_{split}" / "weights" / "best.pt"
                shutil.copy(best, REPO / "weights" / f"{ym['name']}_{split}.pt")

## 3. Inference on the 180 test images -> `submission.csv`

In [ ]:
import torch
from infer import predict_prob
from train import build
from postproc import instances, upsample
from make_sub import validate
from common import rle_counts
from pycocotools import mask as mask_util

dev = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
import cv2
def load_members():
    groups = []
    for m in CFG["unet_members"]:
        ms = []
        for f in range(5):
            net = build(m["enc"], pretrained=False); sd = torch.load(REPO / "weights" / f"{m['name']}_f{f}.pt", map_location="cpu")
            net.load_state_dict({k: v.float() for k, v in sd.items()}); ms.append(net.to(dev).eval())
        groups.append((m["weight"], m.get("res", 1024), ms))
    return groups
GROUPS = load_members()

def full_res(split, stem):
    # native 2048x2048 image: input of the members trained at full resolution
    d = DATA / ("train/train_images" if split == "train" else "test/test_images")
    return cv2.imread(str(d / f"{stem}.jpeg"), cv2.IMREAD_GRAYSCALE)

def ensemble_prob(img_u8, split, stem, groups=None):
    # weighted mean of the members' probability maps, each upsampled to 2048x2048
    groups = groups or GROUPS
    full = full_res(split, stem) if any(r == 2048 for _, r, _ in groups) else None
    tot = sum(w for w, _, _ in groups)
    return sum(w * upsample(predict_prob(ms, full if r == 2048 else img_u8)) for w, r, ms in groups) / tot

P = CFG["postproc"]
def predict_instances(img_u8, stem, yolo_items=None, split="test"):
    prob = ensemble_prob(img_u8, split, stem)
    if CFG.get("yolo") and yolo_items is not None:
        from fuse import fuse
        return fuse(prob, yolo_items, **CFG["yolo"]["fuse"])
    return [m for m, _ in instances(prob, thr=P["thr"], gap=P["gap"], min_area=P["min_area"], min_conf=P["min_conf"])]

In [ ]:
YOLOS = []
if CFG.get("yolo"):
    from ultralytics import YOLO
    from yolo_ens import merge
    YC = CFG["yolo"]
    YOLOS = [YOLO(str(REPO / "weights" / f"{ym['name']}_all.pt")) for ym in YC["models"]]

def yolo_one(model, img_u8):
    r = model.predict(np.repeat(img_u8[:, :, None], 3, 2), imgsz=1024, conf=0.05, iou=0.6,
                      retina_masks=True, verbose=False, max_det=100)[0]
    if r.masks is None:
        return []
    return [(float(c), (m > 0.5).astype(np.uint8)) for m, c in zip(r.masks.data.cpu().numpy(), r.boxes.conf.cpu().numpy())]

def yolo_items_for(img_u8, models=None):
    # instances of every detector, merged by mask IoU (confidence averaged over detectors)
    models = YOLOS if models is None else models
    if not models:
        return None
    views = [yolo_one(m, img_u8) for m in models]
    if len(views) == 1:
        return views[0]
    return [(c, m.astype(np.uint8)) for c, m in merge(views, match=YC["match"], vote=0.5)]

rows = []
test_stems = sorted(p.stem for p in (REPO / "data" / "cache1024" / "test").glob("*.npz"))
if SMOKE:
    test_stems = test_stems[:SMOKE]
for s in test_stems:
    img = np.load(REPO / "data" / "cache1024" / "test" / f"{s}.npz")["img"]
    for k, m in enumerate(predict_instances(img, s, yolo_items_for(img)), 1):
        rows.append({"filament_id": f"{s}_{k}", "segmentation_rle": rle_counts(mask_util.encode(np.asfortranarray(m)))})
sub = pd.DataFrame(rows); validate(sub); sub.to_csv(REPO / "submission.csv", index=False)
print(len(sub), "filaments in", sub.filament_id.str.rsplit("_", n=1).str[0].nunique(), "test images")

## 4. Evaluation on held-out training folds
Each fold model predicts only the observations it never saw (out-of-fold), so every training image gets an
honest prediction. Scores follow the organisers' self-evaluation protocol (per annotator-image ground truth,
IoU > 0.5 matches, Panoptic Quality), plus the IoU / Dice distributions and one-to-many / many-to-one
relations from the rubric. A 95% bootstrap interval over annotator-images is reported.

In [ ]:
from common import load_train, folds, gt_rles_for
from evaluate import overlap_tables, summarize, bootstrap_pq, plot_all
TR = load_train(); stems = sorted(TR); FM = folds(stems)
EVAL_FOLDS = CFG.get("eval_folds", [0, 1, 2, 3, 4])
# fold-specific YOLO models (trained without that fold) for an honest out-of-fold fusion score
FOLD_YOLO = {}
if CFG.get("yolo"):
    from ultralytics import YOLO
    from fuse import fuse
    FOLD_YOLO = {f: [YOLO(str(REPO / "weights" / f"{ym['name']}_f{f}.pt")) for ym in CFG["yolo"]["models"]]
                 for f in CFG["yolo"].get("fold_models", [])}
    EVAL_FOLDS = sorted(FOLD_YOLO) if FOLD_YOLO else EVAL_FOLDS
preds = {}
for f in EVAL_FOLDS:
    fold_groups = [(w, r, [ms[f]]) for w, r, ms in GROUPS]
    for s in [s for s in stems if FM[s] == f][: SMOKE or None]:
        img = np.load(REPO / "data" / "cache1024" / "train" / f"{s}.npz")["img"]
        prob = ensemble_prob(img, "train", s, fold_groups)
        if f in FOLD_YOLO:
            ms = fuse(prob, yolo_items_for(img, FOLD_YOLO[f]), **CFG["yolo"]["fuse"])
        else:
            ms = [m for m, _ in instances(prob, thr=P["thr"], gap=P["gap"], min_area=P["min_area"], min_conf=P["min_conf"])]
        preds[s] = [mask_util.encode(np.asfortranarray(m)) for m in ms]
gt = gt_rles_for(TR, list(preds))
tabs = overlap_tables(gt, preds); S = summarize(tabs)
print(f"OOF PQ {S['pq']:.4f}  (SQ {S['sq']:.3f}, RQ {S['rq']:.3f}; TP {S['tp']} FP {S['fp']} FN {S['fn']})  95% CI {bootstrap_pq(tabs)}")
plot_all(S, "Out-of-fold evaluation"); plt.show()

## 5. Examples on test images

In [ ]:
import cv2
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, s in zip(axes, test_stems[::60]):
    img = cv2.imread(str(DATA / "test" / "test_images" / f"{s}.jpeg"), cv2.IMREAD_GRAYSCALE)
    ov = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    g = sub[sub.filament_id.str.startswith(s)]
    for i, r in enumerate(g.segmentation_rle):
        m = mask_util.decode({"size": [2048, 2048], "counts": r.encode()})
        cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(ov, cnts, -1, tuple(int(x) for x in plt.cm.tab10(i % 10)[:3] * np.array([255, 255, 255])), 3)
    ax.imshow(ov); ax.set_title(f"{s}: {len(g)} filaments"); ax.axis("off")
plt.tight_layout(); plt.show()